## Import Libraries 
Only libraries useful for this project: Matplolib, Pandas, Plotly


In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

# 1. Prepare Data

## 1.1 Import Data

Read the CSV file containing the real estate data from Mexico city. 

In [24]:
df1 = pd.read_csv("data/mexico-city-real-estate-1.csv")
df1.head()

,operation,property_type,place_with_parent_names,lat-lon,price,currency,price_aprox_local_currency,price_aprox_usd,surface_total_in_m2,surface_covered_in_m2,price_usd_per_m2,price_per_m2,floor,rooms,expenses,properati_url
0,sell,apartment,|Miguel Hidalgo|Distrito Federal|México|,"23.634501,-102.552788",5500000.0,MXN,5450245.50,289775.66,NaN,54.0,NaN,101851.851852,NaN,NaN,NaN,http://miguel-hidalgo-df.properati.com.mx/o3zb...
1,sell,house,|Iztapalapa|Distrito Federal|México|,"19.31033,-99.068557",1512000.0,MXN,1498321.97,79661.96,NaN,80.0,NaN,18900.000000,NaN,NaN,NaN,http://iztapalapa.properati.com.mx/q7t0_venta_...
2,sell,apartment,|Tlalpan|Distrito Federal|México|,"19.279771,-99.234597",926667.0,MXN,918284.00,48822.82,NaN,100.0,NaN,9266.670000,NaN,NaN,NaN,http://tlalpan.properati.com.mx/qbi4_venta_dep...
3,sell,apartment,|Miguel Hidalgo|Distrito Federal|México|,"23.634501,-102.552788",6410000.0,MXN,6352013.39,337720.36,NaN,135.0,NaN,47481.481481,NaN,NaN,NaN,http://miguel-hidalgo-df.properati.com.mx/opeq...
4,sell,apartment,|Benito Juárez|Quintana Roo|México|,"21.1902642,-86.8198375",875000.0,USD,16457437.50,875000.00,0.0,263.0,NaN,3326.996198,NaN,NaN,NaN,http://cancun.properati.com.mx/hg4t_venta_depa...


In [25]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 4628 entries, 0 to 4627
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   operation                   4628 non-null   str    
 1   property_type               4628 non-null   str    
 2   place_with_parent_names     4628 non-null   str    
 3   lat-lon                     4144 non-null   str    
 4   price                       4538 non-null   float64
 5   currency                    4538 non-null   str    
 6   price_aprox_local_currency  4538 non-null   float64
 7   price_aprox_usd             4538 non-null   float64
 8   surface_total_in_m2         1668 non-null   float64
 9   surface_covered_in_m2       4436 non-null   float64
 10  price_usd_per_m2            1150 non-null   float64
 11  price_per_m2                4249 non-null   float64
 12  floor                       291 non-null    float64
 13  rooms                       136 non-null    

## 1.2 Data Cleaning 


### Drop NaN values from the dataframe `df1`



In [26]:
# show sum of NaN values in each column
df1.isnull().sum()

operation                        0
property_type                    0
place_with_parent_names          0
lat-lon                        484
price                           90
currency                        90
price_aprox_local_currency      90
price_aprox_usd                 90
surface_total_in_m2           2960
surface_covered_in_m2          192
price_usd_per_m2              3478
price_per_m2                   379
floor                         4337
rooms                         4492
expenses                      4623
properati_url                    0
dtype: int64

In [27]:
# find percentage of null values in each column
df1.isnull().mean() * 100

operation                      0.000000
property_type                  0.000000
place_with_parent_names        0.000000
lat-lon                       10.458081
price                          1.944685
currency                       1.944685
price_aprox_local_currency     1.944685
price_aprox_usd                1.944685
surface_total_in_m2           63.958513
surface_covered_in_m2          4.148660
price_usd_per_m2              75.151253
price_per_m2                   8.189283
floor                         93.712187
rooms                         97.061366
expenses                      99.891962
properati_url                  0.000000
dtype: float64

- We do not drop every row or column that contains any null value because this would remove several valuable apartment price predictors, especially location (lat-lon) and usable area (surface_covered_in_m2). Instead, we drop columns with extremely high percentage of missing values and columns that create target leakage, we then treat the remaining missing values deliberately.
- In this case, a sensible first pass is to drop:
    - `expenses` — 99.9% missing 
    - `rooms` — 97.1% missing 
    - `floor` — 93.7% missing 
    - `price_usd_per_m2` - 75% missing
    - `surface_total_in_m2` - 64% missing
    - `price_usd_per_m2` and `price_per_m2`— these are calculated from price and area, so they leak information about the target price into the model.
        - A feature is leakage if it contains information that would not be available independently at the time you want to predict a listing’s price. Including leakage variables can make validation metrics look unrealistically strong while hurting real-world generalization
    - `price_aprox_usd`, `price_aprox_local_currency`, and possibly `currency` — if price is the target, these are alternate representations or direct transformations of the target, so they should not be predictors - i.e. we have to choose one price target to focus on and remove the rest:
        - all price columns () have same NaN value percentage at 1.944685, therefore they are similar. Since we will be dealing with USD, we'll keep `price_aprox_usd`
    - `properati_url` — this is just a listing identifier/URL, i.e not a stable property characteristic. Same with `operation`
- We keep `lat-lon` and `surface_covered_in_m2`; both contain important signals for apartment price prediction since geographic location and size of apartment are good predictors of apartment prices. Also, their NaN value percentages are low enough to be acceptable since we can later drop them without affecting the predicting power of the source data.

In [28]:
# drop nan columns and leakage columns

columns_to_drop = [
    'expenses',
    'rooms',
    'floor',
    'price_usd_per_m2',
    'price_per_m2',
    'price_aprox_local_currency',
    'currency',
    'surface_total_in_m2',
    'properati_url',
    'operation' 
]

# Drop these columns completely
df1 = df1.drop(columns=columns_to_drop)

df1.head()

,property_type,place_with_parent_names,lat-lon,price,price_aprox_usd,surface_covered_in_m2
0,apartment,|Miguel Hidalgo|Distrito Federal|México|,"23.634501,-102.552788",5500000.0,289775.66,54.0
1,house,|Iztapalapa|Distrito Federal|México|,"19.31033,-99.068557",1512000.0,79661.96,80.0
2,apartment,|Tlalpan|Distrito Federal|México|,"19.279771,-99.234597",926667.0,48822.82,100.0
3,apartment,|Miguel Hidalgo|Distrito Federal|México|,"23.634501,-102.552788",6410000.0,337720.36,135.0
4,apartment,|Benito Juárez|Quintana Roo|México|,"21.1902642,-86.8198375",875000.0,875000.00,263.0


In [29]:
# investigate the number of remaining NaN values
df1.isnull().sum()

property_type                0
place_with_parent_names      0
lat-lon                    484
price                       90
price_aprox_usd             90
surface_covered_in_m2      192
dtype: int64

### Filter data for Apartment Type only

- Since we don't want to arbitrarily remove NaN values, we can further clean the data by focussing on the data we want and then we can later come back to remove any NaN values.
    - We start with focusing on the `property_tupe = apartment` since that is the type of property we will be predicting prices for

In [ ]:


mask_aprt_type = df1["property_type"] =="apartment"
df1 = df1[mask_aprt_type]


In [33]:
df1.head()

,property_type,place_with_parent_names,lat-lon,price,price_aprox_usd,surface_covered_in_m2
0,apartment,|Miguel Hidalgo|Distrito Federal|México|,"23.634501,-102.552788",5500000.0,289775.66,54.0
2,apartment,|Tlalpan|Distrito Federal|México|,"19.279771,-99.234597",926667.0,48822.82,100.0
3,apartment,|Miguel Hidalgo|Distrito Federal|México|,"23.634501,-102.552788",6410000.0,337720.36,135.0
4,apartment,|Benito Juárez|Quintana Roo|México|,"21.1902642,-86.8198375",875000.0,875000.00,263.0
5,apartment,|Miguel Hidalgo|Distrito Federal|México|,"19.432657,-99.177444",4416000.0,232663.51,87.0


### Separate `lat-lon` column into two columns `lat` and `lon`

Ensure that the two columns are of `float` data type

In [ ]:
# Split once, then assign by list index
# split_coords = df1['lat-lon'].str.split(",")

# df1['lat'] = split_coords.str[0]
# df1['lon'] = split_coords.str[1]

df1.head()

In [ ]:
# View the rows that didn't split properly
df1[df1['lon'].isna()]

In [ ]:
# This will output the dataframe so you can see how many columns it actually created
split_df = df1["lat-lon"].str.split(",", expand=True)
split_df.head()

In [ ]:
df1.head()
